In [5]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝


import sys
import importlib

def install_if_missing(package_name: str):
    try:
        importlib.import_module(package_name)
    except ImportError:
        print(f"Installing {package_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_name])


for pkg in ["pandas", "plotly", "ipywidgets"]:
    install_if_missing(pkg)


try:
    import ipywidgets as widgets
    from IPython.display import display, Javascript
    widgets.IntSlider()  # dummy widget to force loading
    display(Javascript("Jupyter.notebook.kernel.execute('from ipywidgets import widgets')"))
except:
    pass

print("All packages ready! Your FPMA dashboard will work perfectly.")


<IPython.core.display.Javascript object>

All packages ready! Your FPMA dashboard will work perfectly.


In [6]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

import pandas as pd

csv_url = "https://raw.githubusercontent.com/tezamo/FPMA/main/data/rawdata.csv"

df = pd.read_csv(csv_url)

df["date"] = pd.to_datetime(df["date"])

# Display the first rows
df.head()


,date,price_usd,commodity_name,country,market,price_type,unit,price_source
0,2022-01-01,0.67,Apples (golden),Republic of Moldova,National Average,RETAIL,Kg,Domestic
1,2022-02-01,0.69,Apples (golden),Republic of Moldova,National Average,RETAIL,Kg,Domestic
2,2022-03-01,0.67,Apples (golden),Republic of Moldova,National Average,RETAIL,Kg,Domestic
3,2022-04-01,0.65,Apples (golden),Republic of Moldova,National Average,RETAIL,Kg,Domestic
4,2022-05-01,0.64,Apples (golden),Republic of Moldova,National Average,RETAIL,Kg,Domestic


In [7]:
# ╔════════════════════════════════════╗
# ║             TEZAMO                 ║
# ╚════════════════════════════════════╝

#=======================
# Import Libraries
#=======================
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

#==================
# create_widget_set
#==================
def create_widget_set(name_prefix):
    # 1. Source
    source_dropdown = widgets.Dropdown(
        options=["Domestic", "International"],
        description=f"{name_prefix} Source:"
    )

    # 2. Commodity
    commodity_dropdown = widgets.Dropdown(description=f"{name_prefix} Commodity:")

    # 3. Price Type
    price_type_dropdown = widgets.Dropdown(description=f"{name_prefix} Price Type:")

    # 4. Country
    country_dropdown = widgets.Dropdown(description=f"{name_prefix} Country:")

    # 5. Market
    market_dropdown = widgets.Dropdown(description=f"{name_prefix} Market:")

    #-------------------------------------------
    # Widget Function
    #-------------------------------------------
    def update_widgets(*args):
        # Filter by source
        df_src = df[df["price_source"] == source_dropdown.value]

        # commodity
        commodities = sorted(df_src["commodity_name"].dropna().unique())
        commodity_dropdown.options = commodities
        if commodity_dropdown.value not in commodities:
            commodity_dropdown.value = commodities[0] if commodities else None

        # Filter by commodity
        df_c = df_src[df_src["commodity_name"] == commodity_dropdown.value] if commodity_dropdown.value else df_src

        # Update price type
        price_types = sorted(df_c["price_type"].dropna().unique())
        price_type_dropdown.options = price_types
        if price_type_dropdown.value not in price_types:
            price_type_dropdown.value = price_types[0] if price_types else None

        # Filter by price type
        df_cp = df_c[df_c["price_type"] == price_type_dropdown.value] if price_type_dropdown.value else df_c

        # Update country
        countries = sorted(df_cp["country"].dropna().unique())
        country_dropdown.options = ["All"] + countries
        if country_dropdown.value not in countries:
            country_dropdown.value = "All"

        # Filter by country
        df_cpc = df_cp if country_dropdown.value == "All" else df_cp[df_cp["country"] == country_dropdown.value]

        # Update market
        markets = sorted(df_cpc["market"].dropna().unique())
        market_dropdown.options = ["All"] + markets
        if market_dropdown.value not in markets:
            market_dropdown.value = "All"

    #-----------------------------
    # output
    #-----------------------------
    source_dropdown.observe(update_widgets, "value")
    commodity_dropdown.observe(update_widgets, "value")
    price_type_dropdown.observe(update_widgets, "value")
    country_dropdown.observe(update_widgets, "value")

    # Initial values
    update_widgets()

    return {
        "source": source_dropdown,
        "commodity": commodity_dropdown,
        "price_type": price_type_dropdown,
        "country": country_dropdown,
        "market": market_dropdown
    }

#==================
# create_plot
#==================
def create_plot(widget_set):

    def plot(source, commodity, price_type, country, market):
        dff = df[df["price_source"] == source]

        # Apply filters
        if commodity != "All":
            dff = dff[dff["commodity_name"] == commodity]
        if price_type != "All":
            dff = dff[dff["price_type"] == price_type]
        if country != "All":
            dff = dff[dff["country"] == country]
        if market != "All":
            dff = dff[dff["market"] == market]

        if dff.empty:
            display("⚠ No data available for this combination.")
            return

        #------------------------------------
        # Setting Unit
        #------------------------------------
        units = dff["unit"].dropna().unique()
        unit_label = units[0] if len(units) == 1 else "Multiple Units"

        #-----------------------------------
        # Legend (Country - Market)
        #----------------------------------
        dff_plot = dff.copy()
        dff_plot["legend_label"] = dff_plot["country"] + " - " + dff_plot["market"]

        #------------------------------
        # Plotting
        #------------------------------
        fig = px.line(
            dff_plot,
            x="date",
            y="price_usd",
            color="legend_label",
            title=f"{commodity} — {price_type} ({unit_label})",
            labels={
                "price_usd": f"Price ({unit_label})",
                "legend_label": "Country - Market"
            }
        )
        fig.update_layout(legend_title="")
        fig.show()

        #--------------------------------
        # Descriptive Analysis
        #--------------------------------
        output_stats = widgets.Output()
        display(output_stats)

        stats_data = []
        df_s = dff.copy()
        num_countries = df_s["country"].nunique()
        data_length_per_country = df_s.groupby("country")["price_usd"].count()
        num_series = len(df_s.groupby(["country", "market", "price_type", "commodity_name"]))
        date_range_overall = {"min": df_s["date"].min(), "max": df_s["date"].max()}
        date_range_per_country = df_s.groupby("country")["date"].agg(["min", "max"])
        null_counts = df_s.isnull().sum()
        gaps = df_s.sort_values("date").groupby("country")["date"].diff().dt.days

        stats_data.append({
            "Selection": f"{commodity} — {price_type} ({unit_label})",
            "Count": len(df_s),
            "Num Countries": num_countries,
            "Data length per country": data_length_per_country.to_dict(),
            "Num Series": num_series,
            "Overall start": date_range_overall["min"],
            "Overall end": date_range_overall["max"],
            "Start/end per country": date_range_per_country.to_dict(orient="index"),
            "Null counts": null_counts.to_dict(),
            "Frequency gaps": gaps.value_counts().to_dict(),
            "Min": df_s["price_usd"].min(),
            "Max": df_s["price_usd"].max(),
            "Mean": df_s["price_usd"].mean(),
            "Median": df_s["price_usd"].median(),
            "Std": df_s["price_usd"].std(),
            "Range": df_s["price_usd"].max() - df_s["price_usd"].min(),
            "% Change": ((df_s["price_usd"].iloc[-1] - df_s["price_usd"].iloc[0]) / df_s["price_usd"].iloc[0]) * 100
        })

        with output_stats:
            output_stats.clear_output()
            for sd in stats_data:
                display(widgets.HTML(f"<h4>{sd['Selection']}</h4>"))
                stats_df = pd.DataFrame({
                    "Metric": [
                        "Count", "Num Countries", "Num Series",
                        "Overall start", "Overall end", "Min", "Max",
                        "Mean", "Median", "Std", "Range", "% Change"
                    ],
                    "Value": [
                        sd["Count"], sd["Num Countries"], sd["Num Series"],
                        sd["Overall start"], sd["Overall end"], sd["Min"], sd["Max"],
                        sd["Mean"], sd["Median"], sd["Std"], sd["Range"], sd["% Change"]
                    ]
                })
                display(stats_df)

                display(widgets.HTML("<b>Data length per country:</b>"))
                display(pd.DataFrame.from_dict(sd["Data length per country"], orient="index", columns=["Count"]))

                display(widgets.HTML("<b>Start/end dates per country:</b>"))
                display(pd.DataFrame.from_dict(sd["Start/end per country"], orient="index"))

                display(widgets.HTML("<b>Null counts:</b>"))
                display(pd.DataFrame.from_dict(sd["Null counts"], orient="index", columns=["Nulls"]))

                display(widgets.HTML("<b>Frequency gaps (days):</b>"))
                display(pd.DataFrame.from_dict(sd["Frequency gaps"], orient="index", columns=["Occurrences"]))

    return plot

#==================
#  Widget Sets
#==================
widgets_A = create_widget_set("A")
widgets_B = create_widget_set("B")

#==================
#  Plots
#==================
plotA = create_plot(widgets_A)
plotB = create_plot(widgets_B)

#==================
#  Panel A
#==================
print("### 📈 Panel A ###")
widgets.interact(
    plotA,
    source=widgets_A["source"],
    commodity=widgets_A["commodity"],
    price_type=widgets_A["price_type"],
    country=widgets_A["country"],
    market=widgets_A["market"]
)

#==================
#  Panel B
#==================
print("### 📈 Panel B ###")
widgets.interact(
    plotB,
    source=widgets_B["source"],
    commodity=widgets_B["commodity"],
    price_type=widgets_B["price_type"],
    country=widgets_B["country"],
    market=widgets_B["market"]
)


### 📈 Panel A ###


interactive(children=(Dropdown(description='A Source:', options=('Domestic', 'International'), value='Domestic…

### 📈 Panel B ###


interactive(children=(Dropdown(description='B Source:', options=('Domestic', 'International'), value='Domestic…

<function __main__.create_plot.<locals>.plot(source, commodity, price_type, country, market)>

In [4]:
#==================
# Libraries
#==================
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

#==================
# create_widget_set
#==================
def create_widget_set(name_prefix):
    source_dropdown = widgets.Dropdown(options=["Domestic", "International"], description=f"{name_prefix} Source:")
    commodity_dropdown = widgets.Dropdown(description=f"{name_prefix} Commodity:")
    price_type_dropdown = widgets.Dropdown(description=f"{name_prefix} Price Type:")
    country_dropdown = widgets.Dropdown(description=f"{name_prefix} Country:")
    market_dropdown = widgets.Dropdown(description=f"{name_prefix} Market:")

    def update_widgets(*args):
        df_src = df[df["price_source"] == source_dropdown.value]

        commodities = sorted(df_src["commodity_name"].dropna().unique())
        commodity_dropdown.options = commodities
        if commodity_dropdown.value not in commodities:
            commodity_dropdown.value = commodities[0] if commodities else None

        df_c = df_src[df_src["commodity_name"] == commodity_dropdown.value] if commodity_dropdown.value else df_src

        price_types = sorted(df_c["price_type"].dropna().unique())
        price_type_dropdown.options = price_types
        if price_type_dropdown.value not in price_types:
            price_type_dropdown.value = price_types[0] if price_types else None

        df_cp = df_c[df_c["price_type"] == price_type_dropdown.value] if price_type_dropdown.value else df_c

        countries = sorted(df_cp["country"].dropna().unique())
        country_dropdown.options = ["All"] + countries
        if country_dropdown.value not in countries:
            country_dropdown.value = "All"

        df_cpc = df_cp if country_dropdown.value == "All" else df_cp[df_cp["country"] == country_dropdown.value]

        markets = sorted(df_cpc["market"].dropna().unique())
        market_dropdown.options = ["All"] + markets
        if market_dropdown.value not in markets:
            market_dropdown.value = "All"

    source_dropdown.observe(update_widgets, "value")
    commodity_dropdown.observe(update_widgets, "value")
    price_type_dropdown.observe(update_widgets, "value")
    country_dropdown.observe(update_widgets, "value")

    update_widgets()

    return {
        "source": source_dropdown,
        "commodity": commodity_dropdown,
        "price_type": price_type_dropdown,
        "country": country_dropdown,
        "market": market_dropdown
    }

#==================
# Plot & Descriptive Analysis
#==================
selections = []
output_plot = widgets.Output()

def update_plot():
    output_plot.clear_output()
    with output_plot:
        if not selections:
            display("⚠ No selections added.")
            return

        fig = go.Figure()
        any_data = False

        #----------------------------------------------------
        # PLOT
        #----------------------------------------------------
        for sel in selections:
            dff = df[df["price_source"] == sel["source"]]

            if sel["commodity"] != "All":
                dff = dff[dff["commodity_name"] == sel["commodity"]]
            if sel["price_type"] != "All":
                dff = dff[dff["price_type"] == sel["price_type"]]
            if sel["country"] != "All":
                dff = dff[dff["country"] == sel["country"]]
            if sel["market"] != "All":
                dff = dff[dff["market"] == sel["market"]]

            if dff.empty:
                print(f"⚠ No data for {sel['label']}")
                continue

            any_data = True

            dff = dff.copy()
            dff["country"] = dff["country"].fillna("Unknown")
            dff["market"] = dff["market"].fillna("Unknown")

            units = dff["unit"].dropna().unique()
            unit_label = units[0] if len(units) == 1 else "Multiple Units"

            dff["legend_label"] = (
                dff["commodity_name"].astype(str) + " • " +
                unit_label + " • " +
                dff["country"].astype(str) + " • " +
                dff["market"].astype(str) + " • " +
                dff["price_type"].astype(str) + " • " +
                dff["price_source"].astype(str)
            )

            for name, group in dff.groupby("legend_label"):
                group_sorted = group.sort_values("date")
                fig.add_trace(
                    go.Scatter(
                        x=group_sorted["date"],
                        y=group_sorted["price_usd"],
                        mode="lines",
                        name=name,
                        hovertemplate="%{x}<br>%{y}<extra></extra>"
                    )
                )

        if not any_data:
            display("⚠ No data available for the added selections.")
            return

        fig.update_layout(
            title="Comined Plot",
            xaxis_title="Date",
            yaxis_title="Price (USD)",
            legend_title=""
        )
        fig.show()

        #----------------------------------------------------
        # DESCRIPTIVE ANALYSIS (TAB)
        #----------------------------------------------------
        tab_titles = []
        tab_contents = []

        for sel in selections:
            dff_s = df[df["price_source"] == sel["source"]]
            if sel["commodity"] != "All":
                dff_s = dff_s[dff_s["commodity_name"] == sel["commodity"]]
            if sel["price_type"] != "All":
                dff_s = dff_s[dff_s["price_type"] == sel["price_type"]]
            if sel["country"] != "All":
                dff_s = dff_s[dff_s["country"] == sel["country"]]
            if sel["market"] != "All":
                dff_s = dff_s[dff_s["market"] == sel["market"]]

            if dff_s.empty:
                continue

            # unit
            unit_options = dff_s["unit"].dropna().unique()
            unit_label_s = unit_options[0] if len(unit_options) == 1 else "Multiple Units"

            # descriptive statistics
            stats = {
                "Count": len(dff_s),
                "Num Countries": dff_s["country"].nunique(),
                "Num Series": len(dff_s.groupby(["country", "market", "price_type", "commodity_name"])),
                "Overall start": dff_s["date"].min(),
                "Overall end": dff_s["date"].max(),
                "Min": dff_s["price_usd"].min(),
                "Max": dff_s["price_usd"].max(),
                "Mean": dff_s["price_usd"].mean(),
                "Median": dff_s["price_usd"].median(),
                "Std": dff_s["price_usd"].std(),
                "Range": dff_s["price_usd"].max() - dff_s["price_usd"].min(),
                "% Change": (
                    (dff_s["price_usd"].iloc[-1] - dff_s["price_usd"].iloc[0]) / dff_s["price_usd"].iloc[0] * 100
                ) if len(dff_s) > 1 else None
            }

            stats_df = pd.DataFrame({"Metric": list(stats.keys()), "Value": list(stats.values())})

            # Additional statistics
            country_count = dff_s.groupby("country")["price_usd"].count().rename("Count").to_frame()
            start_end = dff_s.groupby("country")["date"].agg(["min", "max"])
            nulls = dff_s.isna().sum().rename("Nulls").to_frame()
            gaps_df = dff_s.sort_values("date")["date"].diff().dt.days.value_counts().sort_index().rename("Occurrences").to_frame()

            # output 
            out = widgets.Output()
            with out:
                display(widgets.HTML(
                    f"<h3>{sel['commodity']} • {unit_label_s} • {sel['country']} • "
                    f"{sel['market']} • {sel['price_type']} • {sel['source']}</h3>"
                ))

                display(widgets.HTML("<b>Descriptive Statistics</b>"))
                display(stats_df)

                display(widgets.HTML('<div style="border-top:1px solid #888; margin:10px 0;"></div>'
                                     '<b>Data Length per Country</b>'))
                display(country_count)

                display(widgets.HTML('<div style="border-top:1px solid #888; margin:10px 0;"></div>'
                                     '<b>Start / End Dates per Country</b>'))
                display(start_end)

                display(widgets.HTML('<div style="border-top:1px solid #888; margin:10px 0;"></div>'
                                     '<b>Null Counts</b>'))
                display(nulls)

                display(widgets.HTML('<div style="border-top:1px solid #888; margin:10px 0;"></div>'
                                     '<b>Frequency Gaps (days)</b>'))
                display(gaps_df)

            tab_titles.append(f"{sel['commodity']} ({sel['country']}, {sel['market']})")
            tab_contents.append(out)

        if tab_contents:
            tabs = widgets.Tab(children=tab_contents)
            for i, title in enumerate(tab_titles):
                tabs.set_title(i, title[:25])  # short clean names
            display(tabs)

#==================
# Add / Remove / Clear Buttons
#==================
widget_set = create_widget_set("")
selection_list = selections

btn_add = widgets.Button(description="➕ Add Selection", button_style='success')
btn_remove = widgets.Button(description="❌ Remove Selected", button_style='warning')
btn_clear = widgets.Button(description="🗑 Clear All", button_style='danger')

remove_dropdown = widgets.Dropdown(options=[], description="Remove:", layout=widgets.Layout(width="450px"))

def refresh_remove_dropdown():
    if not selection_list:
        remove_dropdown.options = []
        remove_dropdown.value = None
        return
    labels = [
        f"{i+1}: {sel['commodity']} | {sel['price_type']} | {sel['country']} | {sel['market']} | {sel['source']}"
        for i, sel in enumerate(selection_list)
    ]
    remove_dropdown.options = labels
    remove_dropdown.value = labels[0]

def add_selection(_):
    vals = {k: w.value for k, w in widget_set.items()}
    vals["label"] = f"{vals['commodity']} | {vals['price_type']} | {vals['source']} | {vals['country']} | {vals['market']}"
    selection_list.append(vals)
    refresh_remove_dropdown()
    update_plot()

def remove_selected(_):
    if not selection_list or not remove_dropdown.value:
        return
    idx = remove_dropdown.options.index(remove_dropdown.value)
    del selection_list[idx]
    refresh_remove_dropdown()
    update_plot()

def clear_all(_):
    selection_list.clear()
    refresh_remove_dropdown()
    update_plot()

btn_add.on_click(add_selection)
btn_remove.on_click(remove_selected)
btn_clear.on_click(clear_all)

#==================
# Display
#==================
ui = widgets.VBox(
    list(widget_set.values()) +
    [widgets.HBox([btn_add, btn_remove, btn_clear]), remove_dropdown, output_plot]
)
display(ui)
